# Model Training

## Set Up and Libraries

In [7]:
import os
import numpy as np
import pandas as pd
import pickle
import joblib

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

In [ ]:
# Switch Directories for Google Colab
if os.getcwd() == '/content':
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir("/content/drive/Othercomputers/My MacBook Pro/CreditCardFraudDetection/notebooks")

# Current working directory
print(os.getcwd())

Mounted at /content/drive
/content/drive/Othercomputers/My MacBook Pro/CreditCardFraudDetection/notebooks


## Logistic Regression

In [ ]:
# Load the preprocessed data
with open('../data/preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)
    X_train = data['X_train']
    X_test = data['X_test']
    y_train = data['y_train']
    y_test = data['y_test']

print("Data loaded from preprocessed_data.pkl")

Data loaded from preprocessed_data.pkl


In [ ]:
# Apply Logistic Regression Model
lg = LogisticRegression(solver='liblinear')

# Search for the best class weight instead of forcing 'balanced'
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'class_weight': [None, {0: 1, 1: 5}, {0: 1, 1: 10}, {0: 1, 1: 15}]
}

grid_search = GridSearchCV(estimator=lg, param_grid=param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get the best model and predict
best_model = grid_search.best_estimator_
y_test_pred = best_model.predict(X_test)

print("Best parameters found:", grid_search.best_params_)

Best parameters found: {'C': 10, 'class_weight': {0: 1, 1: 5}, 'penalty': 'l1'}


In [5]:
# Evaluate the base logistic regression model on the testing set
print("Test evaluation - accuracy: {:.3f}, f1: {:.3f}, recall: {:.3f}, precision: {:.3f}"
    .format(accuracy_score(y_test, y_test_pred),
            f1_score(y_test, y_test_pred),
            recall_score(y_test, y_test_pred),
            precision_score(y_test, y_test_pred)))

Test evaluation - accuracy: 0.780, f1: 0.157, recall: 0.596, precision: 0.090


In [6]:
# Try different predicting thresholds
y_probs = best_model.predict_proba(X_test)[:, 1]
for i in range(0, 11):
  threshold = i / 10
  y_pred_tresh = (y_probs >= threshold).astype(int)
  print("Test evaluation (threshold: {:.1f}) - accuracy: {:.3f}, f1: {:.3f}, recall: {:.3f}, precision: {:.3f}"
      .format(threshold,
              accuracy_score(y_test, y_pred_tresh),
              f1_score(y_test, y_pred_tresh),
              recall_score(y_test, y_pred_tresh),
              precision_score(y_test, y_pred_tresh)))

print("\nThe best threshold that fits the project goals is 0.4")
y_pred_tresh = (y_probs >= 0.4).astype(int)
print("Test evaluation (threshold: 0.4) - accuracy: {:.3f}, f1: {:.3f}, recall: {:.3f}, precision: {:.3f}"
    .format(accuracy_score(y_test, y_pred_tresh),
            f1_score(y_test, y_pred_tresh),
            recall_score(y_test, y_pred_tresh),
            precision_score(y_test, y_pred_tresh)))

# Test evaluation (threshold: 0.4) - accuracy: 0.663, f1: 0.125, recall: 0.699, precision: 0.068

Test evaluation (threshold: 0.0) - accuracy: 0.034, f1: 0.066, recall: 1.000, precision: 0.034
Test evaluation (threshold: 0.1) - accuracy: 0.155, f1: 0.073, recall: 0.975, precision: 0.038
Test evaluation (threshold: 0.2) - accuracy: 0.366, f1: 0.089, recall: 0.903, precision: 0.047
Test evaluation (threshold: 0.3) - accuracy: 0.537, f1: 0.106, recall: 0.799, precision: 0.057
Test evaluation (threshold: 0.4) - accuracy: 0.663, f1: 0.125, recall: 0.699, precision: 0.068
Test evaluation (threshold: 0.5) - accuracy: 0.780, f1: 0.157, recall: 0.596, precision: 0.090
Test evaluation (threshold: 0.6) - accuracy: 0.871, f1: 0.207, recall: 0.491, precision: 0.132
Test evaluation (threshold: 0.7) - accuracy: 0.920, f1: 0.252, recall: 0.391, precision: 0.186
Test evaluation (threshold: 0.8) - accuracy: 0.949, f1: 0.288, recall: 0.298, precision: 0.279
Test evaluation (threshold: 0.9) - accuracy: 0.964, f1: 0.273, recall: 0.194, precision: 0.458
Test evaluation (threshold: 1.0) - accuracy: 0.966

In [9]:
# Save the model
joblib.dump(grid_search, '../models/lg.pkl')

# Save the best parameters to a text file
with open('../models/lg_best_params.txt', 'w') as f:
    f.write(str(grid_search.best_params_))